# Career Matching with XGBoost
### AI Career Compass - ML Showcase

This notebook demonstrates how XGBoost can be used for career matching based on user quiz responses.
It is a standalone showcase and is **not connected to the main project**.

**Features used from the quiz:**
- Activity preference (helping, creating, analyzing, building)
- Subject interest (STEM, arts, health, business, social)
- Work environment (office, outdoor, lab, creative, flexible)
- Work style (team, independent, leadership, mixed)
- Motivation (money, impact, creativity, growth, security)
- Education level (high school, trade, bachelor, master, self-taught)

In [ ]:
# Install dependencies
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully!')

## 1. Generate Synthetic Training Data
Since we don't have real user data, we generate synthetic data based on career-trait mappings.

In [ ]:
np.random.seed(42)

# Feature encodings
activity_map    = {'helping': 0, 'creating': 1, 'analyzing': 2, 'building': 3}
subject_map     = {'stem': 0, 'arts': 1, 'health': 2, 'business': 3, 'social': 4}
environment_map = {'office': 0, 'outdoor': 1, 'lab': 2, 'creative': 3, 'flexible': 4}
workstyle_map   = {'team': 0, 'independent': 1, 'leadership': 2, 'mixed': 3}
motivation_map  = {'money': 0, 'impact': 1, 'creativity': 2, 'growth': 3, 'security': 4}
education_map   = {'hs': 0, 'trade': 1, 'bachelor': 2, 'master': 3, 'selflearn': 4}

# Career labels
careers = [
    'Software Engineer', 'Data Scientist', 'UX Designer', 'Product Manager',
    'Doctor', 'Nurse', 'Teacher', 'Psychologist',
    'Financial Analyst', 'Marketing Manager', 'Entrepreneur', 'Architect',
    'Mechanical Engineer', 'Civil Engineer', 'Graphic Designer', 'Content Creator'
]

# Career trait profiles: [activity, subject, environment, workstyle, motivation, education]
career_profiles = {
    'Software Engineer':   [2, 0, 4, 1, 3, 2],
    'Data Scientist':      [2, 0, 2, 1, 3, 3],
    'UX Designer':         [1, 1, 3, 0, 2, 2],
    'Product Manager':     [2, 3, 0, 2, 0, 2],
    'Doctor':              [0, 2, 2, 0, 1, 3],
    'Nurse':               [0, 2, 0, 0, 1, 2],
    'Teacher':             [0, 4, 0, 0, 1, 2],
    'Psychologist':        [0, 4, 0, 1, 1, 3],
    'Financial Analyst':   [2, 3, 0, 1, 0, 2],
    'Marketing Manager':   [1, 3, 0, 2, 0, 2],
    'Entrepreneur':        [1, 3, 4, 2, 2, 4],
    'Architect':           [1, 0, 3, 1, 2, 3],
    'Mechanical Engineer': [3, 0, 2, 0, 3, 2],
    'Civil Engineer':      [3, 0, 1, 0, 3, 2],
    'Graphic Designer':    [1, 1, 3, 1, 2, 4],
    'Content Creator':     [1, 1, 4, 1, 2, 4],
}

# Generate synthetic dataset with noise
n_samples = 2000
X_list, y_list = [], []

for i in range(n_samples):
    career_idx = i % len(careers)
    career = careers[career_idx]
    profile = career_profiles[career]
    
    # Add noise to simulate real user variation
    noise = np.random.randint(-1, 2, size=6)
    sample = np.array(profile) + noise
    
    # Clip to valid ranges
    sample[0] = np.clip(sample[0], 0, 3)  # activity
    sample[1] = np.clip(sample[1], 0, 4)  # subject
    sample[2] = np.clip(sample[2], 0, 4)  # environment
    sample[3] = np.clip(sample[3], 0, 3)  # workstyle
    sample[4] = np.clip(sample[4], 0, 4)  # motivation
    sample[5] = np.clip(sample[5], 0, 4)  # education
    
    X_list.append(sample)
    y_list.append(career_idx)

X = np.array(X_list)
y = np.array(y_list)

# Create DataFrame
df = pd.DataFrame(X, columns=['activity', 'subject', 'environment', 'workstyle', 'motivation', 'education'])
df['career'] = [careers[i] for i in y]

print(f'Dataset shape: {df.shape}')
print(f'\nCareer distribution:')
print(df['career'].value_counts())

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Feature Distributions by Career', fontsize=16, fontweight='bold')

features = ['activity', 'subject', 'environment', 'workstyle', 'motivation', 'education']
feature_labels = [
    ['Helping', 'Creating', 'Analyzing', 'Building'],
    ['STEM', 'Arts', 'Health', 'Business', 'Social'],
    ['Office', 'Outdoor', 'Lab', 'Creative', 'Flexible'],
    ['Team', 'Independent', 'Leadership', 'Mixed'],
    ['Money', 'Impact', 'Creativity', 'Growth', 'Security'],
    ['HS', 'Trade', 'Bachelor', 'Master', 'Self-taught']
]

for idx, (feat, labels) in enumerate(zip(features, feature_labels)):
    ax = axes[idx // 3][idx % 3]
    counts = df[feat].value_counts().sort_index()
    ax.bar(range(len(labels)), [counts.get(i, 0) for i in range(len(labels))], 
           color=plt.cm.Set2(np.linspace(0, 1, len(labels))))
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
    ax.set_title(feat.capitalize(), fontweight='bold')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

## 3. Train XGBoost Model

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Training samples: {X_train.shape[0]}')
print(f'Testing samples:  {X_test.shape[0]}')

# Train XGBoost
model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42
)

model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

# Evaluate
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'\nTest Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)')

In [ ]:
# Cross-validation
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
print(f'Cross-Validation Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'CV Scores: {cv_scores}')

## 4. Classification Report

In [ ]:
print('Classification Report:')
print('=' * 60)
print(classification_report(y_test, y_pred, target_names=careers))

## 5. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=careers, yticklabels=careers)
plt.title('Confusion Matrix - Career Matching XGBoost', fontsize=14, fontweight='bold')
plt.ylabel('Actual Career', fontsize=12)
plt.xlabel('Predicted Career', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Feature Importance

In [ ]:
feature_names = ['Activity Preference', 'Subject Interest', 'Work Environment',
                 'Work Style', 'Motivation', 'Education Level']

importances = model.feature_importances_
sorted_idx = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(feature_names)))
bars = plt.bar(range(len(feature_names)), importances[sorted_idx], color=colors)
plt.xticks(range(len(feature_names)), [feature_names[i] for i in sorted_idx], rotation=30, ha='right')
plt.title('Feature Importance - XGBoost Career Matching', fontsize=14, fontweight='bold')
plt.ylabel('Importance Score')
plt.xlabel('Quiz Features')

for bar, imp in zip(bars, importances[sorted_idx]):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             f'{imp:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print('\nFeature Importance Ranking:')
for i, idx in enumerate(sorted_idx):
    print(f'  {i+1}. {feature_names[idx]}: {importances[idx]:.4f}')

## 7. Predict Career for a New User

In [ ]:
def predict_career(activity, subject, environment, workstyle, motivation, education, top_n=5):
    """
    Predict top N career matches for a user.
    
    Parameters:
    -----------
    activity    : str - 'helping' | 'creating' | 'analyzing' | 'building'
    subject     : str - 'stem' | 'arts' | 'health' | 'business' | 'social'
    environment : str - 'office' | 'outdoor' | 'lab' | 'creative' | 'flexible'
    workstyle   : str - 'team' | 'independent' | 'leadership' | 'mixed'
    motivation  : str - 'money' | 'impact' | 'creativity' | 'growth' | 'security'
    education   : str - 'hs' | 'trade' | 'bachelor' | 'master' | 'selflearn'
    top_n       : int - number of top careers to return
    """
    features = np.array([[
        activity_map[activity],
        subject_map[subject],
        environment_map[environment],
        workstyle_map[workstyle],
        motivation_map[motivation],
        education_map[education]
    ]])
    
    proba = model.predict_proba(features)[0]
    top_indices = np.argsort(proba)[::-1][:top_n]
    
    print(f'\n🎯 Top {top_n} Career Matches:')
    print('=' * 40)
    for rank, idx in enumerate(top_indices, 1):
        print(f'  {rank}. {careers[idx]:<25} {proba[idx]*100:.1f}%')
    
    return [(careers[i], proba[i]) for i in top_indices]


# Example 1: Tech-oriented user
print('Example 1: Tech-oriented user')
predict_career(
    activity='analyzing',
    subject='stem',
    environment='flexible',
    workstyle='independent',
    motivation='growth',
    education='bachelor'
)

# Example 2: Creative user
print('\nExample 2: Creative user')
predict_career(
    activity='creating',
    subject='arts',
    environment='creative',
    workstyle='mixed',
    motivation='creativity',
    education='selflearn'
)

# Example 3: Healthcare-oriented user
print('\nExample 3: Healthcare-oriented user')
predict_career(
    activity='helping',
    subject='health',
    environment='lab',
    workstyle='team',
    motivation='impact',
    education='master'
)

## 8. Model Summary

In [ ]:
print('=' * 50)
print('       XGBoost Career Matching - Summary')
print('=' * 50)
print(f'  Model          : XGBClassifier')
print(f'  Features       : 6 quiz responses')
print(f'  Career Classes : {len(careers)}')
print(f'  Training Size  : {X_train.shape[0]} samples')
print(f'  Test Accuracy  : {accuracy*100:.2f}%')
print(f'  CV Accuracy    : {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%')
print(f'  n_estimators   : 200')
print(f'  max_depth      : 6')
print(f'  learning_rate  : 0.1')
print('=' * 50)
print('\nNote: This is a showcase model using synthetic data.')
print('The actual project uses Gemini LLM for career matching.')